# 09 · Streaming and Debugging

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** Ollama.

A graph that runs for 30 seconds in silence feels broken, and a graph that gives
a wrong answer with no record of *why* is impossible to fix. LangGraph solves
both with one mechanism: **streaming** events out of a running graph, at
whatever level of detail you choose. This notebook covers every stream mode,
then a practical debugging workflow.

**You will learn**
- The stream modes: `values`, `updates`, `messages`, `custom`, `tasks`, `checkpoints`, `debug`
- Streaming LLM **tokens** from inside any node, and filtering by node
- Emitting your own **progress events** with `get_stream_writer`
- Combining modes, streaming from **subgraphs**, and async streaming
- A debugging workflow: draw, stream, inspect state, replay, test, trace (LangSmith)

**Prerequisites:** notebooks 02–05 and 08.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import time, operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer
from common import get_model

model = get_model()

## 1 · Why stream?

| Audience | Needs | Stream mode |
|---|---|---|
| An end user | words appearing as they're generated | `messages` (tokens) |
| A progress UI | "step 2 of 4: researching…" | `updates` or `custom` |
| A developer | full state after each step | `values`, `debug` |
| An operator | what was saved, and when | `checkpoints` |

`invoke()` is simply `stream()` that throws away everything but the final state.
Every mode below is the same run, seen through a different window.

We'll use one small graph throughout: `outline` (an LLM call returning three
points), then `write` (an LLM call turning them into a paragraph).

In [ ]:
class Doc(TypedDict):
    topic: str
    outline: str
    text: str


def outline(state: Doc) -> dict:
    return {'outline': model.invoke(f"3 short bullet points about {state['topic']}. Bullets only.").content}

def write(state: Doc) -> dict:
    return {'text': model.invoke(f"Turn into one short paragraph (max 60 words):\n{state['outline']}").content}


b = StateGraph(Doc)
b.add_node('outline', outline)
b.add_node('write', write)
b.add_edge(START, 'outline')
b.add_edge('outline', 'write')
b.add_edge('write', END)
doc = b.compile()
TOPIC = {'topic': 'why delivery robots need good curb detection'}

## 2 · `values` vs `updates`

- **`values`**: the **whole state** after every superstep. Easy to consume
  (always complete), but heavy when the state is large.
- **`updates`**: **only what each node returned**, keyed by node name. Compact,
  and it tells you *who* changed *what*.

In [ ]:
print('── values ──')
for state in doc.stream(TOPIC, stream_mode='values'):
    print({k: (v[:40] + '…' if isinstance(v, str) and len(v) > 40 else v) for k, v in state.items()})

print('\n── updates ──')
for update in doc.stream(TOPIC, stream_mode='updates'):
    for node, delta in update.items():
        print(f'{node}: changed {list(delta)}')

## 3 · `messages`: token streaming from inside nodes

This is the one users notice. With `stream_mode='messages'`, LangGraph
intercepts **every chat-model call in every node** and streams its tokens out,
even though the nodes call `model.invoke()`, not `model.stream()`. Each item is
a `(chunk, metadata)` pair, and `metadata['langgraph_node']` says which node
the token came from.

Usually you only want to show the final writer's tokens, not the internal
outline, so filter on the node:

In [ ]:
t0 = time.perf_counter()
first = None
for chunk, meta in doc.stream(TOPIC, stream_mode='messages'):
    if meta['langgraph_node'] != 'write':        # hide tokens from the outline step
        continue
    if first is None:
        first = time.perf_counter() - t0
    print(chunk.content, end='', flush=True)
print(f'\n\nfirst visible token after {first:.1f}s, finished after {time.perf_counter() - t0:.1f}s')

The user starts reading after the first token instead of waiting for the
whole run. That's the single biggest improvement to perceived speed.

The metadata carries more than the node name: `langgraph_step`,
`langgraph_path`, the checkpoint namespace, and the model provider. That's
enough to route tokens to the right place in a UI.

## 4 · `custom`: your own progress events

Some progress isn't a state change or a token: "fetched 3 of 10 pages",
"retrying after timeout". Inside any node (or tool), `get_stream_writer()`
returns a function that emits arbitrary data to the `custom` stream:

In [ ]:
class Batch(TypedDict):
    files: list[str]
    done: int


def process(state: Batch) -> dict:
    emit = get_stream_writer()
    for i, name in enumerate(state['files'], 1):
        time.sleep(0.3)                                   # pretend: real work
        emit({'progress': f'{i}/{len(state["files"])}', 'file': name})
    return {'done': len(state['files'])}


b = StateGraph(Batch)
b.add_node('process', process)
b.add_edge(START, 'process')
batch = b.compile()

for event in batch.stream({'files': ['a.pdf', 'b.pdf', 'c.pdf', 'd.pdf'], 'done': 0}, stream_mode='custom'):
    print(event)

If nobody is streaming `custom`, the writer is a no-op, so it's safe to leave
progress events in production code.

## 5 · Several modes at once

Pass a list of modes and each item becomes a `(mode, data)` tuple. A typical UI
subscribes to `updates` (for the progress bar), `custom` (for status text) and
`messages` (for the live answer) in a single run:

In [ ]:
for mode, data in doc.stream(TOPIC, stream_mode=['updates', 'messages']):
    if mode == 'updates':
        print(f"\n[step finished: {list(data)[0]}]")
    elif data[1]['langgraph_node'] == 'write':
        print(data[0].content, end='', flush=True)
print()

**The v2 format.** `stream(..., version='v2')` yields uniform dicts
`{'type': mode, 'ns': namespace, 'data': ...}` instead of mode-dependent shapes.
It's easier to handle generically when you combine many modes:

In [ ]:
for part in doc.stream(TOPIC, stream_mode=['updates'], version='v2'):
    print(part['type'], part['ns'], list(part['data']))

## 6 · Streaming from subgraphs

When a graph contains another graph as a node (notebook 10), you normally see
only the parent's steps. Add `subgraphs=True` and every item is prefixed with a
**namespace**: a tuple that is empty for the parent, and something like
`('child:<id>',)` for events inside the child.

In [ ]:
class Outer(TypedDict):
    topic: str
    outline: str
    text: str

b = StateGraph(Outer)
b.add_node('writer_team', doc)            # the whole `doc` graph as ONE node
b.add_edge(START, 'writer_team')
outer = b.compile()

for namespace, update in outer.stream(TOPIC, stream_mode='updates', subgraphs=True):
    where = 'parent' if not namespace else namespace[0].split(':')[0]
    print(f'{where:<12} {list(update)}')

## 7 · Async streaming

Web servers (FastAPI, etc.) are async. Every method has an async twin:
`ainvoke`, `astream`, `aget_state`. Jupyter supports top-level `await`:

In [ ]:
async def stream_tokens():
    async for chunk, meta in doc.astream(TOPIC, stream_mode='messages'):
        if meta['langgraph_node'] == 'write':
            print(chunk.content, end='', flush=True)
    print()

await stream_tokens()

Use async nodes (`async def node(state)`) when they do I/O. They then run
concurrently on one event loop, and only async nodes support LangGraph's node
`timeout` (notebook 12).

## 8 · Debugging a graph: a workflow

When a graph misbehaves, work from the outside in:

**1. Check the structure.** Is it the graph you think you built?

In [ ]:
print(doc.get_graph().draw_ascii())
print(doc.get_graph().draw_mermaid()[:300], '…')    # paste into mermaid.live, or a Markdown cell

**2. Watch the execution.** The `tasks` mode reports each node as it **starts**
(with its input) and **finishes** (with its result or error). It's the most
direct view of "what ran, with what, in which order":

In [ ]:
for event in doc.stream(TOPIC, stream_mode='tasks'):
    kind = 'start ' if 'input' in event else 'finish'
    detail = list(event['input']) if 'input' in event else (event.get('error') or list(dict(event.get('result') or {})))
    print(f"{kind} {event['name']:<8} {detail}")

**3. Inspect the state at the moment it went wrong.** Compile with a
checkpointer, then read `get_state_history` (notebook 08). You can see exactly
what every node received.

**4. Reproduce cheaply.** Replay from the checkpoint just before the bad step,
or fork it with corrected state to test a fix, without re-running (and
re-paying for) the earlier steps.

**5. Unit-test the node.** Nodes are functions: call the suspect node with the
exact state from the checkpoint and assert on its output. Notebook 12 shows a
testing pattern with a fake model.

**6. Trace in production: LangSmith.** For runs you can't reproduce locally,
you need a recorded trace: every node, every model call with its full prompt
and response, tokens, latency and errors. LangGraph sends traces to
**LangSmith** (LangChain's hosted observability service) when these
environment variables are set, with no code changes:

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...          # from smith.langchain.com
LANGSMITH_PROJECT=phase5-langgraph
```

Put them in `01_LANGGRAPH/.env` to enable tracing for every notebook. The
open-source alternatives (e.g. Langfuse, or anything that speaks
OpenTelemetry) need a callback or exporter instead.

**LangGraph Studio** is a visual debugger that runs locally (`langgraph dev`,
from the `langgraph-cli` package) and shows the graph live, lets you step
through it, edit state and re-run nodes. It needs a small `langgraph.json`
project file; see the LangGraph docs if you want to try it.

In [ ]:
import os
print('LangSmith tracing is', 'ON' if os.getenv('LANGSMITH_TRACING') == 'true' and os.getenv('LANGSMITH_API_KEY')
      else 'off (set LANGSMITH_TRACING and LANGSMITH_API_KEY in .env to enable)')

## 9 · Trade-offs

- **`values` on a big state is expensive.** It copies the whole state every step.
  Prefer `updates` for progress.
- **Token streaming everything is noisy.** Filter by node, or tag models, so
  users see only the answer, not routers and planners.
- **Tracing records prompts and outputs.** That's great for debugging, but it
  may capture personal or confidential data. Decide what's allowed to leave
  your machine before enabling a hosted tracer.

## 10 · Common mistakes

- **Calling `model.stream()` inside nodes to get token streaming.** Not needed:
  `stream_mode='messages'` captures tokens from `invoke()` calls too.
- **Showing every token to the user**, including the router's JSON. Filter on `langgraph_node`.
- **Forgetting `subgraphs=True`** and wondering why the child's steps are invisible.
- **Debugging by adding `print` everywhere.** Use `updates`/`tasks` streams and state history instead;
  they're structured and always available.
- **No tracing in production.** Without it, a user's bad answer usually can't be explained.

## 11 · Check yourself

<details><summary><b>1.</b> What's the difference between <code>values</code> and <code>updates</code>?</summary>

`values` yields the complete state after each superstep. `updates` yields only what each node returned, keyed by node name. Use `updates` for compact progress and to see who changed what.
</details>

<details><summary><b>2.</b> Your nodes call <code>model.invoke()</code>. Can users still see tokens as they're generated?</summary>

Yes. `stream_mode='messages'` intercepts chat-model calls inside nodes and streams their tokens, with metadata saying which node each token came from.
</details>

<details><summary><b>3.</b> How would you show "processed 7 of 20 files" from inside a long node?</summary>

Call `get_stream_writer()` in the node and emit progress dicts, then stream with `stream_mode='custom'` (possibly alongside other modes).
</details>

<details><summary><b>4.</b> A child graph's steps don't appear in your stream. Why?</summary>

By default only the parent's events are streamed. Pass `subgraphs=True`; each item is then prefixed with a namespace identifying which (sub)graph produced it.
</details>

<details><summary><b>5.</b> A user reports a bad answer from yesterday. What do you need to explain it, and how do you get it?</summary>

A trace of that run (every node, prompt, response and error) from tracing such as LangSmith, and/or its checkpoints (via its thread id) to inspect the state at each step and replay from just before the failure.
</details>

## Takeaway

`stream()` shows a running graph at the level of detail you need: `values` or
`updates` for state, `messages` for tokens (filtered by node), `custom` for your
own progress, `tasks`/`checkpoints`/`debug` for developers. Debug from the
outside in (structure, execution, state history, replay, node tests) and turn
on tracing for anything that runs in production.

Next → `10_subgraphs.ipynb`: building big graphs out of smaller ones.